In [ ]:
import requests

def download_n0q_png(dt_str):
    # dt_str format: "YYYYMMDDHHMM"
    date = f"{dt_str[:4]}/{dt_str[4:6]}/{dt_str[6:8]}"
    base_url = f"https://mesonet.agron.iastate.edu/archive/data/{date}/GIS/uscomp/"
    png_url = base_url + f"n0q_{dt_str}.png"
    wld_url = base_url + f"n0q_{dt_str}.wld"

    for url in [png_url, wld_url]:
        fname = url.split("/")[-1]
        r = requests.get(url)
        if r.status_code == 200:
            with open(fname, "wb") as f:
                f.write(r.content)
            print(f"Downloaded {fname}")
        else:
            print(f"Failed to download {fname}: {r.status_code}")

# 示例
download_n0q_png("202505011200")


In [ ]:
import requests
from datetime import datetime, timedelta
import os

def download_n0q_for_hour(date_str, hour):
    base_dt = datetime.strptime(f"{date_str} {hour:02d}:00", "%Y-%m-%d %H:%M")
    for i in range(12): 
        dt = base_dt + timedelta(minutes=5 * i)
        dt_str = dt.strftime("%Y%m%d%H%M")
        date_url = dt.strftime("%Y/%m/%d")
        base_url = f"https://mesonet.agron.iastate.edu/archive/data/{date_url}/GIS/uscomp/"

        for ext in ['png', 'wld']:
            fname = f"n0q_{dt_str}.{ext}"
            url = base_url + fname
            r = requests.get(url)
            if r.status_code == 200:
                with open(fname, 'wb') as f:
                    f.write(r.content)
                print(f"Downloaded {fname}")
            else:
                print(f"Failed to download {fname}")

download_n0q_for_hour("2023-12-22", 0)


In [ ]:
from pathlib import Path

# Notebook that keeps 2023-12-22 radar data, but fetches 2023-12-23 NET echo top images
notebook_code_22_radar_23_net = '''
# Radar data from 2023-12-22; NET Echo Top data forced to 2023-12-23

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import imageio.v2 as imageio
from shapely.geometry import Polygon, box, Point
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.lines import Line2D
from PIL import Image
from io import BytesIO
import requests

# Grid setup
dtw_lat, dtw_lon = 42.2125, -83.3534
radius_deg = 3
grid_step = 0.7

star_fixes = {
    "BONZZ": (-82.7972, 41.7483), "CRAKN": (-82.9405, 41.6730), "CUUGR": (-83.0975, 42.3643),
    "FERRL": (-82.6093, 42.4165), "GRAYT": (-83.6020, 42.9150), "HANBL": (-84.1773, 41.7375),
    "HAYLL": (-84.2975, 41.9662), "HTROD": (-83.3442, 42.0278), "KKISS": (-83.7620, 42.5443),
    "KLYNK": (-82.9888, 41.8793), "LAYKS": (-83.5498, 42.8532), "LECTR": (-84.0217, 41.9183),
    "RKCTY": (-83.9603, 42.6869), "VCTRZ": (-84.0670, 41.9878)
}
ordered_fix_names = ["CRAKN", "BONZZ", "FERRL", "LAYKS", "GRAYT", "RKCTY", "HAYLL", "HANBL"]
tracon_polygon = Polygon([star_fixes[name] for name in ordered_fix_names])
theta = np.linspace(0, 2 * np.pi, 400)
circle_coords = [(dtw_lon + radius_deg * np.cos(t), dtw_lat + radius_deg * np.sin(t)) for t in theta]
circle_polygon = Polygon(circle_coords)
minx, miny, maxx, maxy = circle_polygon.bounds

clipped_cells = []
for lon in np.arange(minx, maxx, grid_step):
    for lat in np.arange(miny, maxy, grid_step):
        cell = box(lon, lat, lon + grid_step, lat + grid_step)
        if circle_polygon.intersects(cell) and not tracon_polygon.contains(cell):
            clipped = cell.intersection(circle_polygon)
            if not clipped.is_empty:
                clipped_cells.append(clipped)

# NET reader
def read_net_echo_top_mask(dstr, grid_polys, threshold_km=10.0):
    date_path = f"{dstr[:4]}/{dstr[4:6]}/{dstr[6:8]}"
    base_url = f"https://mesonet.agron.iastate.edu/archive/data/{date_path}/GIS/uscomp/"
    png_url = base_url + f"net_{dstr}.png"
    wld_url = base_url + f"net_{dstr}.wld"
    try:
        resp = requests.get(png_url)
        if resp.status_code != 200:
            print(f"🔹 Skipping: No NET image for {dstr}")
            return [False] * len(grid_polys)
        net_img = Image.open(BytesIO(resp.content)).convert('L')
        net_array = np.array(net_img)

        wld_txt = requests.get(wld_url).text.strip().splitlines()
        xres = float(wld_txt[0])
        yres = float(wld_txt[3])
        topleft_x = float(wld_txt[4])
        topleft_y = float(wld_txt[5])

        nrows, ncols = net_array.shape
        i_idx, j_idx = np.meshgrid(np.arange(nrows), np.arange(ncols), indexing='ij')
        pixel_lons = topleft_x + j_idx * xres
        pixel_lats = topleft_y + i_idx * yres
        net_km = net_array * 0.5

        valid_mask = net_km >= threshold_km
        valid_lons = pixel_lons[valid_mask]
        valid_lats = pixel_lats[valid_mask]
        points = [Point(lon, lat) for lon, lat in zip(valid_lons, valid_lats)]

        return [any(poly.contains(pt) for pt in points) for poly in grid_polys]

    except Exception as e:
        print(f"❌ Error loading NET for {dstr}: {e}")
        return [False] * len(grid_polys)

# Load 2023-12-22 radar data
df = pd.read_csv("/Users/hoo/Desktop/UMICH/FAAFAST/radar_imgs_20231222_0000_0055/weather_grid_20231222_0000_0055_labeled.csv")
df['time'] = pd.to_datetime(df['time'])
df['is_heavy_cloud'] = df['mean_dbz'] > 20

# Use radar time, but force NET to use 2023-12-23 date
time_points = sorted(df['time'].unique())
has_echo_top_all = []
for t in time_points:
    dstr = pd.Timestamp(t).strftime("20231223%H%M")
    mask = read_net_echo_top_mask(dstr, clipped_cells)
    has_echo_top_all.extend(mask)
df['has_tall_echo_top'] = has_echo_top_all

# Save result
df.to_csv("/Users/hoo/Desktop/UMICH/FAAFAST/radar_imgs_20231222_0000_0055/cloud_echo_net23.csv", index=False)

# Visualization
os.makedirs("frames_echo_overlay", exist_ok=True)
filenames = []
prev_mask = np.zeros(len(clipped_cells), dtype=bool)
deep_blue = "#08306b"
gray = "#636363"

for idx, t in enumerate(time_points):
    frame = df[df['time'] == t].sort_values("grid_id")
    heavy_mask = frame['is_heavy_cloud'].values
    echo_mask = frame['has_tall_echo_top'].values
    combined_mask = np.logical_or(prev_mask, heavy_mask)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(minx - 0.1, maxx + 0.1)
    ax.set_ylim(miny - 0.1, maxy + 0.1)
    ax.set_aspect('equal')
    ax.set_title("Cloud (22) + NET Echo (23)", fontsize=16)

    for i, poly in enumerate(clipped_cells):
        if combined_mask[i]:
            color = gray if echo_mask[i] else deep_blue
            x, y = poly.exterior.xy
            ax.fill(x, y, facecolor=color, edgecolor='lightgray', linewidth=0.4)

    tracon_patch = MplPolygon(tracon_polygon.exterior.coords, closed=True,
                              edgecolor='black', facecolor='none', linewidth=2)
    ax.add_patch(tracon_patch)

    for name, (lon, lat) in star_fixes.items():
        ax.plot(lon, lat, 'ko', markersize=4)
        ax.text(lon + 0.015, lat + 0.01, name, fontsize=7)
    ax.plot(dtw_lon, dtw_lat, 'r*', markersize=12)
    ax.text(dtw_lon + 0.02, dtw_lat, 'KDTW', fontsize=10, fontweight='bold')

    ax.text(0.5, -0.08, f"Time: {pd.Timestamp(t).strftime('%Y-%m-%d %H:%M UTC')}", fontsize=12,
            transform=ax.transAxes, ha='center', bbox=dict(boxstyle="round", facecolor="white"))

    legend_elements = [
        Line2D([0], [0], color=deep_blue, lw=8, label='Heavy Cloud (dBZ > 20)'),
        Line2D([0], [0], color=gray, lw=8, label='Tall Echo Top (from 12/23)')
    ]
    ax.legend(handles=legend_elements, loc='upper right', title="Overlay")

    fig.tight_layout()
    fname = f"frames_echo_overlay/overlay_{idx:03d}.png"
    plt.savefig(fname)
    plt.close()
    filenames.append(fname)
    prev_mask = combined_mask.copy()

imageio.mimsave("Cloud_Overlay_Animation_NET23.gif", [imageio.imread(f) for f in filenames], fps=1)
'''

notebook_path = Path("/mnt/data/radar_22_net_23_combined.ipynb")
with open(notebook_path, "w") as f:
    import nbformat
    nb = nbformat.v4.new_notebook()
    nb.cells.append(nbformat.v4.new_code_cell(notebook_code_22_radar_23_net.strip()))
    nbformat.write(nb, f)

notebook_path
